# Limpieza y Transformación de Datos — Calidad del Aire en México

**Autor:** Alexander Góngora Venegas  
**Curso:** Visualización gráfica para IA – Universidad Iberoamericana León  
**Fuente:** SINAICA – INECC (https://sinaica.inecc.gob.mx/)  

Este notebook documenta el proceso completo de limpieza y transformación de datos.
El mismo proceso está encapsulado de forma reproducible en `src/data_processing.py`.

## Estructura del notebook
1. Configuración e importaciones
2. Generación de datos sintéticos (o carga de datos reales)
3. Exploración inicial (EDA)
4. Limpieza paso a paso
5. Generación de columnas derivadas
6. Agregación y guardado
7. Verificación de resultados

## 1. Configuración e importaciones

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Agrega src/ al path para importar data_processing
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'

print(f'Raíz del proyecto: {ROOT}')
print(f'RAW_DIR existe: {RAW_DIR.exists()}')

## 2. Generación / carga de datos

Si no hay CSVs en `data/raw/`, se generan datos sintéticos que replican los
patrones documentados por SINAICA:
- **Estacionalidad:** PM2.5 alto en invierno (inversiones térmicas), O₃ alto en verano
- **Patrón semanal:** más contaminación martes–jueves, menos sábado–domingo
- **Patrón intradiario:** dos picos (mañana 8h y tarde 19h) para PM2.5/NO₂
- **COVID-19:** caída ~35–45% en NO₂ y CO durante confinamiento (mar–jun 2020)
- **Tendencia:** mejora gradual ~2% anual

In [ ]:
from data_processing import generate_synthetic_raw, load_raw

if not list(RAW_DIR.glob('*.csv')):
    print('No hay CSVs en data/raw/. Generando datos sintéticos...')
    generate_synthetic_raw()
else:
    print(f'CSVs encontrados: {[p.name for p in RAW_DIR.glob("*.csv")]}')

df_raw = load_raw()
print(f'\nShape: {df_raw.shape}')
df_raw.head()

## 3. Exploración inicial (EDA)

Antes de limpiar, entendemos la estructura y la calidad de los datos.

In [ ]:
print('=== Tipos de datos ===')
print(df_raw.dtypes)
print()
print('=== Estadísticas descriptivas ===')
df_raw[['PM25','O3','NO2','CO','SO2']].describe().round(2)

In [ ]:
# Porcentaje de valores nulos por columna y ciudad
poll_cols = ['PM25', 'O3', 'NO2', 'CO', 'SO2']
null_pct = df_raw.groupby('ciudad')[poll_cols].apply(lambda g: g.isna().mean() * 100)
print('Porcentaje de NaN por ciudad y contaminante:')
null_pct.round(2)

In [ ]:
# Distribución por ciudad
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, city in enumerate(sorted(df_raw['ciudad'].unique())):
    sub = df_raw[df_raw['ciudad'] == city]['PM25'].dropna()
    axes[i].hist(sub, bins=60, color='#e8765a', edgecolor='none', alpha=0.85)
    axes[i].set_title(city, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('PM2.5 (μg/m³)')
    axes[i].set_ylabel('Frecuencia')
    axes[i].axvline(sub.median(), color='#c0392b', linestyle='--', label=f'Mediana: {sub.median():.1f}')
    axes[i].legend(fontsize=9)

plt.suptitle('Distribución horaria de PM2.5 por ciudad (2019–2024)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Limpieza paso a paso

Criterios de limpieza adoptados (con justificación):

| Paso | Criterio | Justificación |
|------|----------|---------------|
| 1 | Eliminar registros con `fecha_hora` NaT | Sin timestamp no hay información útil |
| 2 | Valores negativos → NaN | Físicamente imposibles (concentración ≥ 0) |
| 3 | Valores > límite máximo → NaN | Falla de sensor o dato corrupto |
| 4 | Interpolar huecos ≤ 3 horas | Fallas cortas de sensor se pueden recuperar linealmente |
| 5 | Descartar filas con >3 NaN simultáneos | Registro mayoritariamente vacío no aporta información |

In [ ]:
from data_processing import clean, MAX_VALUES

n_orig = len(df_raw)
df_clean = clean(df_raw)
n_clean = len(df_clean)

print(f'Filas originales:  {n_orig:,}')
print(f'Filas tras limpieza: {n_clean:,}')
print(f'Filas eliminadas: {n_orig - n_clean:,} ({(n_orig - n_clean)/n_orig*100:.2f}%)')

In [ ]:
# Verificar que no quedan negativos ni valores extremos
for poll, maxval in MAX_VALUES.items():
    if poll in df_clean.columns:
        n_neg = (df_clean[poll] < 0).sum()
        n_max = (df_clean[poll] > maxval).sum()
        print(f'{poll}: negativos={n_neg}, sobre límite={n_max}')

## 5. Columnas derivadas

Se añaden variables temporales para facilitar el análisis de patrones:

In [ ]:
# Verificar columnas derivadas generadas por clean()
time_cols = ['año', 'mes', 'dia_semana', 'nombre_dia', 'hora', 'fecha', 'semana', 'estacion_año']
print('Columnas derivadas presentes:')
for col in time_cols:
    present = col in df_clean.columns
    sample = df_clean[col].iloc[0] if present else 'N/A'
    print(f'  {col}: {present} → ejemplo: {sample}')

In [ ]:
# Patrón semanal promedio de PM2.5 (todas las ciudades)
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_es = {'Monday':'Lun','Tuesday':'Mar','Wednesday':'Mié',
          'Thursday':'Jue','Friday':'Vie','Saturday':'Sáb','Sunday':'Dom'}

weekly_pm25 = df_clean.groupby('nombre_dia')['PM25'].mean().reindex(day_order)
weekly_pm25.index = [day_es[d] for d in weekly_pm25.index]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e8765a' if i < 5 else '#aecde8' for i in range(7)]
ax.bar(weekly_pm25.index, weekly_pm25.values, color=colors, edgecolor='none')
ax.set_title('Promedio de PM2.5 por día de la semana (todas las ciudades)', fontsize=12)
ax.set_ylabel('PM2.5 (μg/m³)')
ax.axhline(weekly_pm25.mean(), color='#c0392b', linestyle='--', alpha=0.7, label='Media')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nDía más contaminado: {weekly_pm25.idxmax()} ({weekly_pm25.max():.2f} μg/m³)')
print(f'Día más limpio:      {weekly_pm25.idxmin()} ({weekly_pm25.min():.2f} μg/m³)')

## 6. Agregación y guardado

In [ ]:
from data_processing import build_aggregates, save

daily, weekly_pattern, monthly = build_aggregates(df_clean)
save(daily, weekly_pattern, monthly)

print('Archivos generados en data/processed/:')
for f in sorted(PROCESSED_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:30s} {size_kb:8.1f} KB')

## 7. Verificación de resultados

In [ ]:
# Carga y verifica los archivos procesados
daily_check = pd.read_parquet(PROCESSED_DIR / 'daily_avg.parquet')
weekly_check = pd.read_parquet(PROCESSED_DIR / 'weekly_pattern.parquet')
monthly_check = pd.read_parquet(PROCESSED_DIR / 'monthly_series.parquet')

print('=== daily_avg ===')
print(f'Shape: {daily_check.shape}')
print(f'Ciudades: {sorted(daily_check["ciudad"].unique())}')
print(f'Rango: {daily_check["fecha"].min()} → {daily_check["fecha"].max()}')
print()
print('=== weekly_pattern ===')
print(f'Shape: {weekly_check.shape}')
print()
print('=== monthly_series ===')
print(f'Shape: {monthly_check.shape}')
print()
print('✅ Todos los archivos generados correctamente.')

In [ ]:
# Gráfica de verificación: serie mensual por ciudad
fig, ax = plt.subplots(figsize=(12, 5))
palette = ['#e8765a','#2196F3','#4CAF50','#FF9800','#9C27B0','#795548']

for i, city in enumerate(sorted(monthly_check['ciudad'].unique())):
    sub = monthly_check[monthly_check['ciudad'] == city]
    ax.plot(sub['fecha_mes'], sub['PM25'], label=city,
            color=palette[i % len(palette)], linewidth=1.8)

# Franja COVID
ax.axvspan(pd.Timestamp('2020-03-23'), pd.Timestamp('2020-06-01'),
           alpha=0.15, color='steelblue', label='Confinamiento COVID')

ax.set_title('PM2.5 mensual por ciudad (2019–2024)', fontsize=13)
ax.set_ylabel('PM2.5 (μg/m³)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()